# Thailand EPPO — Table 2.3-4 exploration

Validates the two raw Excel sources before Phase 1c (scraper/processor):

- **Historical:** `data/raw/thailand/T02_03_04-1.xlsx` (monthly 1986–2024)
- **Current:** `data/raw/thailand/T02_03_04.xlsx` (annual + Q1 averages + monthly Apr 2025–Mar 2026)

## Conventions (aligned with `reference/product_map.csv`)

- Unified **primary** products only: REGULAR, PREMIUM, HSD, LSD, KEROSENE, J.P., FUEL OIL, LPG.
- Dropped from the series: GASOHOL, U95, fuel-oil grades, and parent totals (GASOLINE, DIESEL, TOTAL).
- Historical `JP` / `FUELOIL` normalize via [`reference/eppo.py`](../reference/eppo.py).
- **2025 Q1 gap:** Jan–Mar 2025 filled with the 2025 Q1 3-month average (`is_provisional=True`).

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "reference" / "eppo.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "reference" / "eppo.py").exists():
            return candidate / "country_oil_scraper"
    raise FileNotFoundError("Could not find country_oil_scraper project root")


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from reference.eppo import (
    EPPO_AGENCY_SOURCE,
    EPPO_DATASET_SOURCE,
    EPPO_METRIC_TYPE,
    EPPO_UNIT_NATIVE,
    eppo_primary_product_names,
    is_eppo_unified_primary,
    normalize_eppo_product_name,
)
from reference.loaders import (
    canonical_category,
    canonical_metric,
    canonical_subcategory,
    is_primary,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "thailand"
HIST_PATH = RAW_DIR / "T02_03_04-1.xlsx"
CURR_PATH = RAW_DIR / "T02_03_04.xlsx"

PRIMARY_PRODUCTS = eppo_primary_product_names()
print("Project root:", PROJECT_ROOT)
print("Unified primaries:", PRIMARY_PRODUCTS)
print("Metric:", canonical_metric("Sale of Petroleum Products", EPPO_DATASET_SOURCE))

## 2. Parse historical workbook (wide → long)

In [ ]:
# Column layout (row 5–6): gasoline total/regular/premium, kerosene,
# diesel total/hsd/lsd, jp, fueloil, lpg, country total.
HIST_PRODUCT_COLS = [
    (1, "TOTAL", "GASOLINE"),
    (2, "REGULAR", "GASOLINE"),
    (3, "PREMIUM", "GASOLINE"),
    (4, "KEROSENE", None),
    (5, "TOTAL", "DIESEL"),
    (6, "HSD", "DIESEL"),
    (7, "LSD", "DIESEL"),
    (8, "JP", None),
    (9, "FUELOIL", None),
    (10, "LPG", None),
    (11, "TOTAL", "COUNTRY"),
]
MONTHS = [
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC",
]


def parse_historical_eppo(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="T2.3-4M", header=None)
    rows = []
    i = 0
    while i < len(raw):
        year_val = raw.iloc[i, 0]
        if isinstance(year_val, (int, float)) and 1980 < year_val < 2030:
            year = int(year_val)
            # Layout: year row, YEAR header row, sub-header row, then JAN..DEC
            data_start = i + 3
            for offset, month in enumerate(MONTHS):
                r = data_start + offset
                if r >= len(raw):
                    break
                if str(raw.iloc[r, 0]).strip().upper() != month:
                    continue
                for col_idx, sub, parent in HIST_PRODUCT_COLS:
                    val = raw.iloc[r, col_idx]
                    if pd.isna(val):
                        continue
                    native = normalize_eppo_product_name(sub)
                    if parent in ("GASOLINE", "DIESEL") and sub == "TOTAL":
                        continue  # skip parent totals
                    if parent == "COUNTRY":
                        continue
                    if not is_eppo_unified_primary(native):
                        continue
                    rows.append({
                        "date": pd.Timestamp(year=year, month=offset + 1, day=1),
                        "product_native": native,
                        "value": float(val),
                        "is_provisional": False,
                        "source_file": path.name,
                    })
            # Advance past JAN..DEC (12 rows) + annual TOTAL row
            i = data_start + len(MONTHS) + 1
        else:
            i += 1
    df = pd.DataFrame(rows)
    return df.sort_values(["date", "product_native"]).reset_index(drop=True)


hist = parse_historical_eppo(HIST_PATH)
print(f"Historical: {len(hist):,} rows")
print(f"  Dates: {hist['date'].min().date()} → {hist['date'].max().date()}")
print(f"  Products: {sorted(hist['product_native'].unique())}")
hist.head(8)

## 3. Parse current workbook (row products × column time)

In [ ]:
# Current file layout (0-based column indices from inspection):
#   1–3: annual 2023–2025
#   4–6: 3-MONTH Q1 averages 2024–2026
#   7–15: monthly 2025 Apr–Dec
#   16–18: monthly 2026 Jan–Mar
CURR_ANNUAL_COLS = {2023: 1, 2024: 2, 2025: 3}
CURR_Q1_COLS = {2024: 4, 2025: 5, 2026: 6}
CURR_MONTHLY_2025 = [
    (7, 2025, 4), (8, 2025, 5), (9, 2025, 6), (10, 2025, 7), (11, 2025, 8),
    (12, 2025, 9), (13, 2025, 10), (14, 2025, 11), (15, 2025, 12),
]
CURR_MONTHLY_2026 = [(16, 2026, 1), (17, 2026, 2), (18, 2026, 3)]
DATA_START_ROW = 5  # first product row (GASOLINE)


def _parse_current_rows(raw: pd.DataFrame, col_specs, *, provisional: bool) -> list[dict]:
    out = []
    for row_idx in range(DATA_START_ROW, len(raw)):
        label = raw.iloc[row_idx, 0]
        if pd.isna(label) or str(label).strip() == "":
            continue
        native = normalize_eppo_product_name(label)
        if not is_eppo_unified_primary(native):
            continue
        for col, year, month in col_specs:
            val = raw.iloc[row_idx, col]
            if pd.isna(val):
                continue
            out.append({
                "date": pd.Timestamp(year=year, month=month, day=1),
                "product_native": native,
                "value": float(val),
                "is_provisional": provisional,
                "source_file": CURR_PATH.name,
            })
    return out


def parse_current_eppo(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.read_excel(path, sheet_name="tab55", header=None)
    monthly_specs = CURR_MONTHLY_2025 + CURR_MONTHLY_2026
    monthly_rows = _parse_current_rows(raw, monthly_specs, provisional=False)

    # Q1 2025 imputation: repeat 2025 Q1 average for Jan–Mar
    q1_rows = []
    q1_col = CURR_Q1_COLS[2025]
    for row_idx in range(DATA_START_ROW, len(raw)):
        label = raw.iloc[row_idx, 0]
        if pd.isna(label) or str(label).strip() == "":
            continue
        native = normalize_eppo_product_name(label)
        if not is_eppo_unified_primary(native):
            continue
        val = raw.iloc[row_idx, q1_col]
        if pd.isna(val):
            continue
        for month in (1, 2, 3):
            q1_rows.append({
                "date": pd.Timestamp(year=2025, month=month, day=1),
                "product_native": native,
                "value": float(val),
                "is_provisional": True,
                "source_file": path.name,
            })

    annual_rows = []
    for year, col in CURR_ANNUAL_COLS.items():
        annual_rows.extend(
            _parse_current_rows(
                raw,
                [(col, year, 1)],  # placeholder month; marked below
                provisional=False,
            )
        )
    # Fix annual dates to year-end marker for validation only
    annual = pd.DataFrame(annual_rows)
    if not annual.empty:
        annual["date"] = pd.to_datetime(annual["date"].dt.year.astype(str) + "-12-31")
        annual["observation_type"] = "annual_avg"

    monthly = pd.DataFrame(monthly_rows + q1_rows)
    return monthly.sort_values(["date", "product_native"]).reset_index(drop=True), annual


curr_monthly, curr_annual = parse_current_eppo(CURR_PATH)
print(f"Current monthly+imputed: {len(curr_monthly):,} rows")
print(f"  Dates: {curr_monthly['date'].min().date()} → {curr_monthly['date'].max().date()}")
print(f"  Provisional rows: {curr_monthly['is_provisional'].sum():,}")
curr_monthly.tail(8)

## 4. Product map coverage

In [ ]:
raw_curr = pd.read_excel(CURR_PATH, sheet_name="tab55", header=None)
raw_labels = [
    str(raw_curr.iloc[r, 0]).strip()
    for r in range(DATA_START_ROW, len(raw_curr))
    if pd.notna(raw_curr.iloc[r, 0]) and str(raw_curr.iloc[r, 0]).strip()
]

coverage = pd.DataFrame({
    "raw_label": raw_labels,
    "normalized": [normalize_eppo_product_name(x) for x in raw_labels],
})
coverage["unified_primary"] = coverage["normalized"].map(is_eppo_unified_primary)
coverage["product_canonical"] = coverage["normalized"].map(
    lambda n: canonical_subcategory(n, EPPO_AGENCY_SOURCE) if n else None
)
coverage["category"] = coverage["normalized"].map(
    lambda n: canonical_category(n, EPPO_AGENCY_SOURCE) if n else None
)

print("Dropped detail / aggregate rows:")
display(coverage[~coverage["unified_primary"]])
print("\nUnified primaries:")
display(coverage[coverage["unified_primary"]])

## 5. Overlap validation (2023–2024 annual vs historical TOTAL rows)

In [ ]:
def annual_from_historical(df: pd.DataFrame, year: int) -> pd.Series:
    """Mean of monthly values in that calendar year (barrels/day)."""
    sl = df[df["date"].dt.year == year]
    return sl.groupby("product_native")["value"].mean()


def annual_from_current(annual_df: pd.DataFrame, year: int) -> pd.Series:
    sl = annual_df[annual_df["date"].dt.year == year]
    return sl.set_index("product_native")["value"]


for year in (2023, 2024):
    h = annual_from_historical(hist, year)
    c = annual_from_current(curr_annual, year)
    cmp = pd.DataFrame({"historical_mean": h, "current_annual_col": c}).dropna(how="any")
    cmp["diff"] = cmp["current_annual_col"] - cmp["historical_mean"]
    cmp["pct_diff"] = 100 * cmp["diff"] / cmp["historical_mean"]
    print(f"\n=== {year} annual comparison (barrels/day) ===")
    display(cmp.round(2))

# Spot-check Dec 2024 gasoline total vs current annual col
dec24 = hist[(hist["date"] == "2024-12-01") & (hist["product_native"] == " PREMIUM")]["value"].iloc[0]
ann24_premium = curr_annual[(curr_annual["product_native"] == " PREMIUM") & (curr_annual["date"].dt.year == 2024)]["value"].iloc[0]
print(f"\n2024 PREMIUM: historical Dec={dec24:,.2f}  current annual={ann24_premium:,.2f}")

## 6. Stitch preview (HSD + country total proxy)

In [ ]:
stitched = pd.concat([
    hist[hist["date"] <= "2024-12-01"],
    curr_monthly[curr_monthly["date"] >= "2025-01-01"],
], ignore_index=True)
stitched = stitched.sort_values(["product_native", "date"])

print(f"Stitched rows: {len(stitched):,}")
print(f"  Range: {stitched['date'].min().date()} → {stitched['date'].max().date()}")
print(f"  Provisional: {stitched['is_provisional'].sum():,}")

for product in [" HSD", " PREMIUM", "FUEL OIL"]:
    s = stitched[stitched["product_native"] == product]
    fig = go.Figure()
    obs = s[~s["is_provisional"]]
    prov = s[s["is_provisional"]]
    fig.add_trace(go.Scatter(x=obs["date"], y=obs["value"], mode="lines", name="observed"))
    if not prov.empty:
        fig.add_trace(go.Scatter(
            x=prov["date"], y=prov["value"], mode="lines+markers",
            name="Q1 2025 imputed", line=dict(dash="dash"),
        ))
    fig.update_layout(
        title=f"{product.strip()} — EPPO sales (bbl/d)",
        xaxis_title="Date", yaxis_title="Barrels / day", height=380,
    )
    fig.show()

## 7. Q1 2025 imputation sanity check

In [ ]:
q1 = curr_monthly[(curr_monthly["date"].between("2025-01-01", "2025-03-01"))]
apr = curr_monthly[curr_monthly["date"] == "2025-04-01"]

chk = q1.groupby("product_native")["value"].agg(["min", "max", "count"])
chk["flat_q1"] = chk["min"] == chk["max"]
print("Q1 2025 should be flat (3 identical months per product):")
display(chk)

join = q1.drop_duplicates("product_native").merge(
    apr, on="product_native", suffixes=("_q1", "_apr")
)
join["apr_vs_q1_pct"] = 100 * (join["value_apr"] - join["value_q1"]) / join["value_q1"]
print("\nApr 2025 vs Q1 average (% change) — large jumps flag imputation bias:")
display(join[["product_native", "value_q1", "value_apr", "apr_vs_q1_pct"]].round(2))

## 8. Summary

In [ ]:
print("Phase 1a checklist")
print("  [x] Historical parsed:", hist["date"].min().date(), "→", hist["date"].max().date())
print("  [x] Current parsed:", curr_monthly["date"].min().date(), "→", curr_monthly["date"].max().date())
print("  [x] Q1 2025 imputed rows:", int(curr_monthly["is_provisional"].sum()))
print("  [x] Unified primaries:", len(PRIMARY_PRODUCTS))
print("  [x] metric_types:", EPPO_DATASET_SOURCE, "→", EPPO_METRIC_TYPE)
print("\nNext: Phase 1c — move parsers into scrapers/thailand_eppo.py + processor parquet.")